# Phase 6: Feature Engineering for Stage 1 Model

This notebook creates structured features for the Stage 1 fraud detection model.

**What this notebook does:**
- Loads the cleaned dataset from Phase 5
- Creates numeric, binary, and engineered features
- Preserves categorical features for later encoding
- Saves the feature table for modeling

**What this notebook does NOT do:**
- Text embeddings (that's Stage 2, Phase 9)
- Model training (that's Phase 7)
- Heavy normalization (kept simple)

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Add src to path so we can import our modules
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Import our feature engineering module
from feature_engineering import (
    load_cleaned_dataset,
    create_feature_table,
    run_feature_quality_checks,
    save_feature_table,
    summarize_engineered_features,
    get_feature_columns,
    NUMERIC_FEATURES,
    BINARY_FEATURES,
    PRECOMPUTED_FEATURES,
    ENGINEERED_FEATURES,
    CATEGORICAL_FEATURES,
    META_COLUMNS,
)

print("Imports successful!")
print(f"Project root: {PROJECT_ROOT}")

## 2. Load Cleaned Dataset

In [ ]:
# Load the cleaned dataset from Phase 5
df_cleaned = load_cleaned_dataset()

print(f"\nDataset shape: {df_cleaned.shape}")
print(f"Columns: {len(df_cleaned.columns)}")

In [ ]:
# Preview the data
df_cleaned.head(3)

## 3. Review Base Columns for Features

Let's look at the columns we'll use to create features.

In [ ]:
print("=" * 50)
print("NUMERIC FEATURES (direct from data)")
print("=" * 50)
for col in NUMERIC_FEATURES:
    if col in df_cleaned.columns:
        print(f"  {col}: {df_cleaned[col].dtype}")
    else:
        print(f"  {col}: MISSING!")

In [ ]:
print("=" * 50)
print("BINARY FEATURES (direct from data)")
print("=" * 50)
for col in BINARY_FEATURES:
    if col in df_cleaned.columns:
        value_counts = df_cleaned[col].value_counts().to_dict()
        print(f"  {col}: {value_counts}")
    else:
        print(f"  {col}: MISSING!")

In [ ]:
print("=" * 50)
print("CATEGORICAL FEATURES")
print("=" * 50)
for col in CATEGORICAL_FEATURES:
    if col in df_cleaned.columns:
        n_unique = df_cleaned[col].nunique()
        print(f"  {col}: {n_unique} unique values")
    else:
        print(f"  {col}: MISSING!")

## 4. Create Engineered Features Step by Step

Let's create each engineered feature manually to understand the logic.

### 4.1 Income-to-Age Ratio

**Formula:** `annual_income / age`

**Purpose:** Higher ratios might indicate unrealistic income claims for the applicant's age.

In [ ]:
# Create income_age_ratio
income_age_ratio = df_cleaned["annual_income"] / df_cleaned["age"]

print("Income-to-Age Ratio Statistics:")
print(income_age_ratio.describe())

# Show by fraud label
print("\nMean by fraud_label:")
print(df_cleaned.groupby("fraud_label").apply(
    lambda x: (x["annual_income"] / x["age"]).mean()
))

### 4.2 Minimum Tenure

**Formula:** `min(months_at_address, months_at_employer)`

**Purpose:** Low tenure at both address and employer may indicate higher risk.

In [ ]:
# Create tenure_min
tenure_min = df_cleaned[["months_at_address", "months_at_employer"]].min(axis=1)

print("Minimum Tenure Statistics:")
print(tenure_min.describe())

# Show by fraud label
print("\nMean by fraud_label:")
print(pd.DataFrame({
    "tenure_min": tenure_min,
    "fraud_label": df_cleaned["fraud_label"]
}).groupby("fraud_label")["tenure_min"].mean())

### 4.3 Night Application Flag

**Rule:** `1 if application_hour in [0, 1, 2, 3, 4, 5] else 0`

**Purpose:** Applications submitted late night / early morning may be suspicious.

In [ ]:
# Create night_application_flag
NIGHT_HOURS = [0, 1, 2, 3, 4, 5]
night_application_flag = df_cleaned["application_hour"].isin(NIGHT_HOURS).astype(int)

print(f"Night Application Flag (hours {NIGHT_HOURS}):")
print(night_application_flag.value_counts())

# Show rate by fraud label
print("\nRate by fraud_label:")
print(pd.DataFrame({
    "night_flag": night_application_flag,
    "fraud_label": df_cleaned["fraud_label"]
}).groupby("fraud_label")["night_flag"].mean())

### 4.4 High Device Velocity Flag

**Rule:** `1 if num_prev_apps_same_device_7d >= 3 else 0`

**Purpose:** Multiple applications from the same device in a short period is a fraud signal.

In [ ]:
# Create high_device_velocity_flag
THRESHOLD = 3
high_device_velocity_flag = (df_cleaned["num_prev_apps_same_device_7d"] >= THRESHOLD).astype(int)

print(f"High Device Velocity Flag (>= {THRESHOLD} apps in 7 days):")
print(high_device_velocity_flag.value_counts())

# Show rate by fraud label
print("\nRate by fraud_label:")
print(pd.DataFrame({
    "device_velocity_flag": high_device_velocity_flag,
    "fraud_label": df_cleaned["fraud_label"]
}).groupby("fraud_label")["device_velocity_flag"].mean())

### 4.5 High Identity Reuse Flag

**Rule:** `1 if any of (email_30d, phone_30d, address_30d) >= 2 else 0`

**Purpose:** Reuse of identity elements across applications indicates coordinated attacks or synthetic identity fraud.

In [ ]:
# Create high_identity_reuse_flag
REUSE_THRESHOLD = 2
reuse_cols = [
    "num_prev_apps_same_email_30d",
    "num_prev_apps_same_phone_30d",
    "num_prev_apps_same_address_30d",
]
max_reuse = df_cleaned[reuse_cols].max(axis=1)
high_identity_reuse_flag = (max_reuse >= REUSE_THRESHOLD).astype(int)

print(f"High Identity Reuse Flag (any reuse >= {REUSE_THRESHOLD}):")
print(high_identity_reuse_flag.value_counts())

# Show rate by fraud label
print("\nRate by fraud_label:")
print(pd.DataFrame({
    "identity_reuse_flag": high_identity_reuse_flag,
    "fraud_label": df_cleaned["fraud_label"]
}).groupby("fraud_label")["identity_reuse_flag"].mean())

## 5. Create Complete Feature Table

Now let's use the module to create the complete feature table.

In [ ]:
# Create the complete feature table
df_features = create_feature_table(df_cleaned)

In [ ]:
# Preview the feature table
print(f"Feature table shape: {df_features.shape}")
df_features.head()

In [ ]:
# List all columns
print("All columns in feature table:")
for i, col in enumerate(df_features.columns, 1):
    print(f"  {i:2d}. {col}")

## 6. Summary Statistics for Engineered Features

In [ ]:
# Get summary of engineered features
summary = summarize_engineered_features(df_features)
summary

In [ ]:
# Detailed stats for all numeric features
all_numeric = NUMERIC_FEATURES + BINARY_FEATURES + PRECOMPUTED_FEATURES + ENGINEERED_FEATURES
df_features[all_numeric].describe().T

## 7. Feature Distributions by Fraud Label

Let's check if our engineered features show expected patterns.

In [ ]:
# Compare means by fraud label for all numeric features
print("Mean values by fraud_label:")
print("=" * 60)

comparison = df_features.groupby("fraud_label")[all_numeric].mean().T
comparison.columns = ["Legitimate (0)", "Fraud (1)"]
comparison["Diff"] = comparison["Fraud (1)"] - comparison["Legitimate (0)"]
comparison["Ratio"] = comparison["Fraud (1)"] / comparison["Legitimate (0)"]

comparison.round(3)

In [ ]:
# Binary flag rates by fraud label
print("Binary Flag Rates by Fraud Label:")
print("=" * 60)

binary_flags = BINARY_FEATURES + ["night_application_flag", "high_device_velocity_flag", "high_identity_reuse_flag"]

for flag in binary_flags:
    rates = df_features.groupby("fraud_label")[flag].mean()
    print(f"\n{flag}:")
    print(f"  Legitimate: {rates[0]:.1%}")
    print(f"  Fraud:      {rates[1]:.1%}")
    print(f"  Ratio:      {rates[1]/rates[0]:.2f}x" if rates[0] > 0 else "  Ratio: N/A")

## 8. Missingness Check

In [ ]:
# Check for missing values
missing = df_features.isna().sum()
missing_pct = 100 * missing / len(df_features)

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
})

# Show only columns with missing values (if any)
cols_with_missing = missing_df[missing_df["Missing Count"] > 0]

if len(cols_with_missing) > 0:
    print("Columns with missing values:")
    print(cols_with_missing)
else:
    print("No missing values in any column!")

## 9. Run Quality Checks

In [ ]:
# Run all quality checks
original_row_count = len(df_cleaned)
run_feature_quality_checks(df_features, original_row_count)

## 10. Save Feature Table

In [ ]:
# Save the feature table
save_feature_table(df_features)

## 11. Final Summary

### Feature Table Contents

In [ ]:
print("=" * 60)
print("PHASE 6 COMPLETE: FEATURE ENGINEERING SUMMARY")
print("=" * 60)

print(f"\nDataset: {len(df_features):,} rows, {len(df_features.columns)} columns")

print(f"\nMetadata columns ({len(META_COLUMNS)}):")
for col in META_COLUMNS:
    print(f"  - {col}")

print(f"\nNumeric features ({len(NUMERIC_FEATURES)}):")
for col in NUMERIC_FEATURES:
    print(f"  - {col}")

print(f"\nBinary features ({len(BINARY_FEATURES)}):")
for col in BINARY_FEATURES:
    print(f"  - {col}")

print(f"\nPre-computed features ({len(PRECOMPUTED_FEATURES)}):")
for col in PRECOMPUTED_FEATURES:
    print(f"  - {col}")

print(f"\nEngineered features ({len(ENGINEERED_FEATURES)}):")
for col in ENGINEERED_FEATURES:
    print(f"  - {col}")

print(f"\nCategorical features ({len(CATEGORICAL_FEATURES)}):")
for col in CATEGORICAL_FEATURES:
    print(f"  - {col}")

print(f"\nTotal modeling features: {len(get_feature_columns())}")

In [ ]:
# Final preview
print("\nSample rows from feature table:")
df_features.head()

### Key Observations

Based on the feature analysis:

1. **Engineered flags show expected patterns:**
   - `high_device_velocity_flag` is higher in fraud
   - `high_identity_reuse_flag` is higher in fraud
   - `night_application_flag` may show slight difference

2. **Tenure features are discriminative:**
   - `tenure_min` is lower for fraud cases
   - `months_at_address` and `months_at_employer` both lower in fraud

3. **Ready for Stage 1 modeling:**
   - All features are created and validated
   - No missing values
   - Categorical features preserved for encoding in modeling phase